In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import random

C:\Users\maath\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# Exemple : récupérer une page HTML 

url = 'https://books.toscrape.com'   # Site dédié à la pratique du scraping
 
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
 
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()   # Lève une erreur si status != 200
    print(" Requête réussie — Status : {response.status_code}")
    print(" Taille de la réponse : {len(response.text):,} caractères")
 
    # Parser le HTML avec BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
 
    # Extraire les titres
    titres = soup.find_all('h3')
    print("\n {len(titres)} éléments trouvés sur la page")
    print("Premier élément : {titres[0].get_text(strip=True)[:60]}")
 
except requests.exceptions.RequestException as e:
    print("Connexion impossible : {e}")
    print("(Mode données générées activé)\n")

 Requête réussie — Status : {response.status_code}
 Taille de la réponse : {len(response.text):,} caractères

 {len(titres)} éléments trouvés sur la page
Premier élément : {titres[0].get_text(strip=True)[:60]}


In [3]:
#Exemple : parser un HTML directement
 
html_exemple = """
<div class="offre">
    <h2>Data Analyst</h2>
    <span class="entreprise">TechCorp Paris</span>
    <span class="lieu">Paris, France</span>
    <span class="salaire">45 000 - 55 000 EUR</span>
    <span class="tags">Python, SQL, Power BI</span>
</div>
<div class="offre">
    <h2>Data Scientist</h2>
    <span class="entreprise">DataLab Lyon</span>
    <span class="lieu">Lyon, France</span>
    <span class="salaire">50 000 - 65 000 EUR</span>
    <span class="tags">Python, ML, Seaborn</span>
</div>
"""
 
soup_demo = BeautifulSoup(html_exemple, 'html.parser')
offres_html = soup_demo.find_all('div', class_='offre')
 
print("\n Parsing HTML — {len(offres_html)} offres extraites :")
for o in offres_html:
    titre = o.find('h2').get_text()
    lieu  = o.find('span', class_='lieu').get_text()
    print(f"   {titre} — {lieu}")


 Parsing HTML — {len(offres_html)} offres extraites :
   Data Analyst — Paris, France
   Data Scientist — Lyon, France


In [4]:
#  PARTIE 2 — DONNÉES RÉALISTES (simulation API jobs)
 
random.seed(42)
 
postes     = ['Data Analyst', 'Data Scientist', 'Data Engineer',
              'ML Engineer', 'BI Developer', 'Analytics Engineer']
villes     = ['Paris', 'Lyon', 'Bordeaux', 'Toulouse', 'Nantes', 'Remote']
contrats   = ['CDI', 'CDI', 'CDI', 'CDD', 'Alternance', 'Stage', 'Freelance']
competences = ['Python', 'SQL', 'Power BI', 'Tableau', 'Spark',
               'Docker', 'dbt', 'Airflow', 'Scikit-learn', 'TensorFlow']
niveaux    = ['Junior (0-2 ans)', 'Mid (2-5 ans)', 'Senior (5+ ans)']
 
salaires = {
    'Junior (0-2 ans)':   (32000, 42000),
    'Mid (2-5 ans)':      (42000, 58000),
    'Senior (5+ ans)':    (58000, 80000),
}
 
offres = []
date_base = datetime.now()
 
for i in range(120):
    niveau  = random.choice(niveaux)
    sal_min = salaires[niveau][0]
    sal_max = salaires[niveau][1]
    salaire = random.randint(sal_min, sal_max)
 
    offres.append({
        'id':          i + 1,
        'poste':       random.choice(postes),
        'ville':       random.choice(villes),
        'contrat':     random.choice(contrats),
        'niveau':      niveau,
        'salaire':     salaire,
        'competences': ', '.join(random.sample(competences, random.randint(2, 4))),
        'date':        (date_base - timedelta(days=random.randint(0, 30)))
                        .strftime('%Y-%m-%d'),
    })
 
# Sauvegarder en JSON (comme une vraie API)
with open('offres_emploi.json', 'w', encoding='utf-8') as f:
    json.dump(offres, f, ensure_ascii=False, indent=2)
 
print("\n {len(offres)} offres sauvegardées → offres_emploi.json")


 {len(offres)} offres sauvegardées → offres_emploi.json


In [5]:
#  PARTIE 3 — ANALYSE AVEC PANDAS
 
# Charger le JSON comme si c'était une API
with open('offres_emploi.json', encoding='utf-8') as f:
    data = json.load(f)
 
df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])
 
print("\n Dataset : {df.shape[0]} offres × {df.shape[1]} colonnes")
 
print("\n Postes les plus demandés :")
print(df['poste'].value_counts())
 
print("\n Villes :")
print(df['ville'].value_counts())
 
print("\n Salaire moyen par poste :")
print(df.groupby('poste')['salaire'].mean().sort_values(ascending=False).round(0))


 Dataset : {df.shape[0]} offres × {df.shape[1]} colonnes

 Postes les plus demandés :
poste
Data Analyst          26
Data Engineer         24
BI Developer          22
Analytics Engineer    18
ML Engineer           17
Data Scientist        13
Name: count, dtype: int64

 Villes :
ville
Paris       24
Toulouse    23
Nantes      22
Remote      21
Bordeaux    17
Lyon        13
Name: count, dtype: int64

 Salaire moyen par poste :
poste
Data Scientist        57634.0
Data Engineer         54633.0
Analytics Engineer    52768.0
BI Developer          52334.0
ML Engineer           50124.0
Data Analyst          47732.0
Name: salaire, dtype: float64


In [6]:
#  PARTIE 4 — VISUALISATION
 
BG     = '#0F172A'
CARD   = '#1E293B'
COLORS = ['#00FF94', '#00E0FF', '#A78BFA', '#FFD700', '#FF6B6B', '#F97316']
TEXTE  = '#E2E8F0'
GRIS   = '#94A3B8'
 
def style(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=GRIS, labelsize=9)
    ax.xaxis.label.set_color(GRIS)
    ax.yaxis.label.set_color(GRIS)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#334155')
 
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor(BG)
fig.suptitle('Analyse Offres Emploi Data — Jour 5/30',
             fontsize=14, fontweight='bold', color=TEXTE, y=0.98)
 
# Graphe 1 — Offres par poste
ax1 = axes[0, 0]; style(ax1)
counts = df['poste'].value_counts()
bars = ax1.barh(counts.index, counts.values, color=COLORS, edgecolor=BG, height=0.6)
ax1.set_title('Offres par Poste', color=TEXTE, fontweight='bold')
ax1.set_xlim(0, counts.max() * 1.2)
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.3, bar.get_y() + bar.get_height()/2,
             str(int(w)), va='center', color=GRIS, fontsize=9)
 
# Graphe 2 — Répartition par ville
ax2 = axes[0, 1]; style(ax2)
villes_count = df['ville'].value_counts()
ax2.pie(villes_count.values, labels=villes_count.index,
        autopct='%1.0f%%', colors=COLORS, startangle=90,
        wedgeprops=dict(edgecolor=BG, linewidth=1.5))
ax2.set_title('Répartition par Ville', color=TEXTE, fontweight='bold')
ax2.set_facecolor(CARD)
 
# Graphe 3 — Salaire moyen par poste
ax3 = axes[1, 0]; style(ax3)
sal_moyen = df.groupby('poste')['salaire'].mean().sort_values()
bars3 = ax3.barh(sal_moyen.index, sal_moyen.values, color=COLORS, edgecolor=BG, height=0.6)
ax3.set_title('Salaire Moyen par Poste (€)', color=TEXTE, fontweight='bold')
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1000:.0f}k€'))
ax3.set_xlim(0, sal_moyen.max() * 1.25)
for bar in bars3:
    w = bar.get_width()
    ax3.text(w + 200, bar.get_y() + bar.get_height()/2,
             f'{w/1000:.0f}k€', va='center', color=GRIS, fontsize=9)
 
# Graphe 4 — Offres par type de contrat
ax4 = axes[1, 1]; style(ax4)
contrats_count = df['contrat'].value_counts()
bars4 = ax4.bar(contrats_count.index, contrats_count.values,
                color=COLORS, edgecolor=BG, linewidth=1.5)
ax4.set_title('Offres par Contrat', color=TEXTE, fontweight='bold')
ax4.set_ylabel('Nombre d\'offres', color=GRIS)
for bar in bars4:
    h = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2, h + 0.3,
             str(int(h)), ha='center', color=TEXTE, fontsize=9)
 
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('dashboard_offres_emploi.png', dpi=150, bbox_inches='tight', facecolor=BG)
print('Dashboard exporté → dashboard_offres_emploi.png')
plt.show()

Dashboard exporté → dashboard_offres_emploi.png


C:\Users\maath\AppData\Local\Temp\ipykernel_4732\1998872139.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
